# **Statistical Learning for Automation Systems**

Author: Matteo Cercola

___

# Principal Component Analysis with Python - Case Study



## PCA Review

PCA consists in a linear transformation of the available input data and attempts to find out which features mostly explain the variance in the data. 

<img src='PCA.png' />

## Import Data Analysis and Visualization Libraries

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

%matplotlib inline

## Import Dataset

We'll work with the built-in Breast Cancer dataset from Scikit Learn since it has a very high number of features.

In [ ]:
from sklearn.datasets import load_breast_cancer
cancer = load_breast_cancer()                       # Assign breast cancer dataset to variable 'cancer'
cancer                                              # 'cancer' is a variable of type 'Dictionary'

In [ ]:
# Look at keys of dictionary
cancer.keys()

In [ ]:
# Inspect a field
# print(cancer['DESCR'])

# Inspect output
cancer['target_names']

# Inspect output value
cancer['target']

# Generate DataFrame
df = pd.DataFrame(cancer['data'], columns = cancer['feature_names'])
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

## Principal Component Analysis

As intuitive, it is difficult to visualize high dimensional data, so we can use PCA for dimensionality reduction. 

Before doing this, we need to remove bias and scale our data so that each feature has unit variance.

### Data Pre-Processing

Scikit-Learn function named StandardScaler() standardizes features by removing the bias and scaling to unit variance (it automatically performs input centering and input normalization). 

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
scaler.fit(df)
scaled_data = scaler.transform(df)

### Singular Value Decomposition

SVD takes the data and decomposes it into 3 matrices s.t.
$$
X = U \Sigma V^*
$$
$\Sigma$ is diagonal (rectangular) with dimensions $m \times n$ and each diagonal element represent the direction of maximum "stretching"

In simple terms, SVD decomposes the transformation induced by $X$ into two rotations $U$ and $V^*$ and a scaling $\Sigma$

<img src='SVD.png' width="1000" />


In [ ]:
# Compute SVD for scaled_data matrix, obtaining as output the matrices for its factorization: scaled_data = U * diag(sigma) * V'
U, sigma, VT = np.linalg.svd(scaled_data, full_matrices=True)

# sigma is an array of dimension 30 containing the ordered singular values of scaled_data matrix
print(sigma)
print('Size of sigma', sigma.shape)

# Sigma is the diagonal matrix of dimension 30 * 30 with ordered singular values on the main diagonal
Sigma = np.diag(sigma)           # Diagonal matrix 
Sigma.shape

In [ ]:
# Plot ordered singular values
plt.figure(figsize = (8,4))
plt.scatter(range(1, len(sigma)+1), sigma)
plt.grid()
plt.ylabel('Singular values')
plt.xlabel('Feature index i')
plt.title('Singular Values Plot')

Most of the information is contained in the first two principal components. We can use PCA to find the first two principal components and visualize the data in this new, two-dimensional space, with a single scatter plot.

Note that the columns of $V$ are the principal directions (eigenvectors), so they indicate the most important directions for the data distribution. Thus, to compress the data, we should project $X$ on the columns of $V$ 

In [ ]:
# Project scaled_data into the first two principal components
X_reduced = np.dot(scaled_data, VT.T[:,:2])     # We are computing X * V along only the first elements of V
X_reduced.shape

Sigma.shape

### PCA Algorithm

Instantiate a PCA object, find the principal components using the fit() method, then apply the rotation and dimensionality reduction by calling transform().

It is also possible to specify how many components we want to keep when creating the PCA object.

In [ ]:
from sklearn.decomposition import PCA                 # PCA() import
pca = PCA(n_components = 2)                           # PCA() object instantiation (specifying the number of desired components)
pca.fit(scaled_data)                                  # fit(X) method fits the model with X (it finds the principal components)

Now we can transform data to the first 2 principal components.

In [ ]:
# transform(X) method applies dimensionality reduction to X
x_pca = pca.transform(scaled_data)                     
print(scaled_data.shape)                                      # Dimensions of standardized dataset before PCA application
print(x_pca.shape)                                            # Dimensions of dataset after PCA application

In [ ]:
x_pca_df = pd.DataFrame(x_pca, columns=['PC1', 'PC2'])        # Pandas Dataframe containing data in the reduced Feature Space
x_pca_df.head()

In [ ]:
x_pca_df.info()

Observation: We've reduced 30 dimensions to just 2! Let's plot these two dimensions out!

## Plot Data in the Reduced Feature Space

In [ ]:
plt.figure(figsize=(8,4))

sns.scatterplot(x=x_pca[:, 0], y=x_pca[:, 1], hue=cancer['target'], s=60)


plt.xlabel('First Principal Component')
plt.ylabel('Second Principal Component')
plt.title('Input Data Scatterplot in the Reduced Feature Space')
plt.xlim([-10, 20])
plt.ylim([-10, 15])

By using these two principal components, we can easily separate the two classes (target = 0, target = 1).

## Interpreting the components 

Unfortunately, with this great power of dimensionality reduction, comes the cost of being able to easily understand what these components represent.

The components correspond to linear combinations of the original features. The components themselves are stored as an attribute of the fitted PCA object:

In [ ]:
pca.components_

In this numpy matrix array, each row represents a principal component and each column relates back to the original features. 

We can visualize this relationship with a heatmap.

In [ ]:
df_comp = pd.DataFrame(pca.components_ , columns = cancer['feature_names'])                      # Create a Pandas DataFrame containing the two principal components
df_comp                                                      # Show df_comp

In [ ]:
# Heatmap Plot
plt.figure(figsize=(8,4))
sns.heatmap(df_comp , cmap='plasma')

# Coefficients of first direction
plt.figure(figsize=(12,6))
plt.bar(np.arange(len(df_comp.iloc[0,:])), df_comp.iloc[0,:])
plt.xticks(np.arange(len(df_comp.iloc[0,:])), df_comp.columns, rotation=90)
plt.show()

# Coefficients of second direction
plt.figure(figsize=(12,6))
plt.bar(np.arange(len(df_comp.iloc[1,:])), df_comp.iloc[1,:])
plt.xticks(np.arange(len(df_comp.iloc[1,:])), df_comp.columns, rotation=90)
plt.show()

This heatmap and the color bar basically represent the correlation between the various features and the principal component itself.

## Can we use PCA in a way other than simply visualizing higher-dimensional data?

We can use PCA to reduce the size of our model to simplify its structure without loosing excessive information!

When you perform Principal Component Analysis (PCA), the goal is to find new orthogonal directions (principal components) that capture as much of the total variance in your dataset as possible.

Each principal component $i$ has an associated variance (also called an eigenvalue of the covariance matrix), denoted by $\lambda_i$ and correspond to the square of the principal components

In [ ]:
# Transform the data into the new space
pca_result = np.dot(scaled_data, VT.T)

# Compute the explained variance
explained_variance = (sigma ** 2) / (scaled_data.shape[1] - 1)  # Variance explained by each principal component

# Compute total variance
total_variance = np.sum(explained_variance)                     # Total variance in the dataset

# Compute explained variance ratio
explained_variance_ratio = explained_variance / total_variance # Proportion of variance explained by each principal component

# Compute cumulative variance ratio
cumulative_variance_ratio = np.cumsum(explained_variance_ratio) # Cumulative proportion of variance explained by the principal components

#Create the scree plot
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(explained_variance) + 1), cumulative_variance_ratio, marker='o', label='Cumulative Variance Ratio')
plt.axhline(y=0.95, color='red', linestyle='--', label='95% Variance Threshold')
plt.title('Scree Plot')
plt.xlabel('Principal Component')
plt.ylabel('Cumulative Variance Ratio')
plt.xticks(range(1, len(explained_variance) + 1))
plt.grid()
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
K = np.argmax(cumulative_variance_ratio >= 0.95) + 1
K # Number of principal components required to retain at least 95% of the variance in the dataset

This plots is useful to have an heuristic to identify the K value for the PCA

In [ ]:
X_reduced = np.dot(scaled_data, VT.T[:,:K])          # Project scaled_data into the first K principal components
X_reduced.shape

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_reduced, cancer['target'], test_size=0.2, random_state=42)  # Split the dataset into training and testing sets
X_train.shape, y_train.shape, X_test.shape, y_test.shape

In [ ]:
from sklearn.linear_model import LogisticRegression                      # Import Logistic Regression

In [ ]:
logmodel = LogisticRegression()                                          # Logistic Regression model instantiation
logmodel.fit(X_train, y_train)                                           # Model fitting

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

predictions = logmodel.predict(X_test)                                   # Predictions on the test set
print(classification_report(y_test, predictions))                        # Classification report

What was the accuracy of the NN model? Again 98%

## Conclusion

The presented case study is an example of application of PCA to a dataset characterized by a very high-dimensional feature space. 

- First, by projecting high-dimensional data onto the first two principal components, PCA provided an effective 2D representation that preserved most of the variance, allowing us to visually identify clusters, trends, and potential outliers in the dataset.
- Second, we applied PCA as a preprocessing step prior to training a machine learning model, showing that reducing the number of features can significantly decrease model complexity and computational cost while maintaining comparable predictive performance.


Overall, the study highlights how PCA helps uncover the underlying structure of high-dimensional data and improves model efficiency, making it a fundamental tool in exploratory data analysis and modern data-driven pipelines.


The drawback relates to the poor interpretability of what principal components represent.